In [1]:
import numpy as np 
import numpy.ma as ma 
import pandas as pd 
import tensorflow as tf 
from tensorflow import keras 
from sklearn.preprocessing import StandardScaler, MinMaxScaler 
from sklearn.model_selection import train_test_split 
import tabulate 
pd.set_option("display.precision", 1)

## 2 - Movie ratings dataaset 

The data set is derived from the [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) dataset. 

[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

The original dataset has roughly 9000 movies rated by 600 users with ratings on a scale of 0.5 to 5 in 0.5 step increments. The dataset has been reduced in size to focus on movies from the years since 2000 and popular genres. The reduced dataset has $n_u = 397$ users, $n_m= 847$ movies and 25521 ratings. For each movie, the dataset provides a movie title, release date, and one or more genres. For example "Toy Story 3" was released in 2010 and has several genres: "Adventure|Animation|Children|Comedy|Fantasy". This dataset contains little information about users other than their ratings. This dataset is used to create training vectors for the neural networks described below. 
Let's learn a bit more about this data set. The table below shows the top 10 movies ranked by the number of ratings. These movies also happen to have high average ratings. How many of these movies have you watched? 

In [2]:
top10_df = pd.read_csv("./data/content_top10_df.csv") 
bygenre_df = pd.read_csv("./data/content_bygenre_df.csv") 
top10_df

,movie id,num ratings,ave rating,title,genres
0,4993,198,4.1,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
1,5952,188,4.0,"Lord of the Rings: The Two Towers, The",Adventure|Fantasy
2,7153,185,4.1,"Lord of the Rings: The Return of the King, The",Action|Adventure|Drama|Fantasy
3,4306,170,3.9,Shrek,Adventure|Animation|Children|Comedy|Fantasy|Ro...
4,58559,149,4.2,"Dark Knight, The",Action|Crime|Drama
5,6539,149,3.8,Pirates of the Caribbean: The Curse of the Bla...,Action|Adventure|Comedy|Fantasy
6,79132,143,4.1,Inception,Action|Crime|Drama|Mystery|Sci-Fi|Thriller
7,6377,141,4.0,Finding Nemo,Adventure|Animation|Children|Comedy
8,4886,132,3.9,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy
9,7361,131,4.2,Eternal Sunshine of the Spotless Mind,Drama|Romance|Sci-Fi


In [3]:
bygenre_df

,genre,num movies,ave rating/genre,ratings per genre
0,Action,321,3.4,10377
1,Adventure,234,3.4,8785
2,Animation,76,3.6,2588
3,Children,69,3.4,2472
4,Comedy,326,3.4,8911
5,Crime,139,3.5,4671
6,Documentary,13,3.8,280
7,Drama,342,3.6,10201
8,Fantasy,124,3.4,4468
9,Horror,56,3.2,1345


## 3 - Content-based filtering with a neural network 

In the collaborative filtering lab, you generated two vectors, a user vector and an item/movie vector whose dot product would predict a rating. The vectors were derived solely from the ratings.   

Content-based filtering also generates a user and movie feature vector but recognizes there may be other information available about the user and/or movie that may improve the prediction. The additional information is provided to a neural network which then generates the user and movie vector as shown below.

In [14]:
from numpy import genfromtxt
from collections import defaultdict
import csv
import pickle

In [15]:
def load_data(): 
    item_train = genfromtxt('./data/content_item_train.csv', delimiter = ',')
    user_train = genfromtxt('./data/content_user_train.csv', delimiter = ',') 
    y_train = genfromtxt('./data/content_y_train.csv', delimiter = ',') 

    with open('./data/content_item_train_header.txt', newline='') as f: 
        item_features = list(csv.reader(f))[0]

    with open('./data/content_user_train_header.txt', newline='') as f: 
        user_features = list(csv.reader(f))[0]

    item_vecs = genfromtxt('./data/content_item_vecs.csv', delimiter = ',') 

    movie_dict = defaultdict(dict)
    count = 0

    #    with open('./data/movies.csv', newline='') as csvfile:
    with open('./data/content_movie_list.csv', newline='') as csvfile:
        reader = csv.reader(csvfile, delimiter=',', quotechar='"')
        for line in reader:
            if count == 0:
                count += 1  #skip header
                #print(line) print
            else:
                count += 1
                movie_id = int(line[0])
                movie_dict[movie_id]["title"] = line[1]
                movie_dict[movie_id]["genres"] = line[2]

    with open('./data/content_user_to_genre.pickle', 'rb') as f:
        user_to_genre = pickle.load(f)

    return(item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre)

In [16]:
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

In [18]:
num_user_features = user_train.shape[1] - 3 # remove userid, rating count and ave rating during training 
num_item_features = item_train.shape[1] - 1 # remove move id at train time 
uvs = 3 # user genre vector start 
ivs = 3 # item genre vector start 
u_s = 3 # start of columns to use in training, user 
i_s = 1 # start of columns to use in training, items

print(f"Number of training vectors: {len(item_train)}") 

Number of training vectors: 50884


In [19]:
print(f"y_train[:5]: {y_train[:5]}") 

y_train[:5]: [4.  3.5 4.  4.  4.5]


## 3.2 Preparing the training data

In [20]:
# scale training data 

item_train_unscaled = item_train 
user_train_unscaled = user_train 
y_train_unscaled = y_train 

scalerItem = StandardScaler()
scalerItem.fit(item_train) 
item_train = scalerItem.transform(item_train) 

scalerUser = StandardScaler()
scalerUser.fit(user_train) 
user_train = scalerUser.transform(user_train) 

scalerTarget = MinMaxScaler((-1, 1)) 
scalerTarget.fit(y_train.reshape(-1, 1))
y_train = scalerTarget.transform(y_train.reshape(-1, 1))

print(np.allclose(item_train_unscaled, scalerItem.inverse_transform(item_train)))
print(np.allclose(user_train_unscaled, scalerUser.inverse_transform(user_train)))

True
True


## 4 - Neural Network for content-based filtering 

Now, let's construct a neural network as described in the figure above. It will have two networks that are combined by a dot product. You will construct the two networks. In this example, they will be identical. Note that these networks do not need to be the same. If the user content was substantially larger than the movie content, you might elect to increase the complexity of the user network relative to the movie network.

In [32]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test data shape: {item_test.shape}")

movie/item training data shape: (40707, 17)
movie/item test data shape: (10177, 17)


In [22]:
num_outputs = 32 
tf.random.set_seed(1) 

user_NN = tf.keras.models.Sequential(
    [
        tf.keras.layers.Dense(256, activation = 'relu'), 
        tf.keras.layers.Dense(128, activation = 'relu'), 
        tf.keras.layers.Dense(num_outputs),
    ]
)

item_NN = tf.keras.models.Sequential(
    [
        tf.keras.layers.Dense(256, activation = 'relu'), 
        tf.keras.layers.Dense(128, activation = 'relu'), 
        tf.keras.layers.Dense(num_outputs),
    ]
)

In [33]:
import tensorflow as tf

input_user = tf.keras.layers.Input(shape=(num_user_features,))
vu_raw = user_NN(input_user)

# Wrap l2_normalize inside a Lambda layer
vu = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(vu_raw)

input_item = tf.keras.layers.Input(shape=(num_item_features,))
vm = item_NN(input_item)

output = tf.keras.layers.Dot(axes=-1)([vu, vm])

model = tf.keras.Model(inputs=[input_user, input_item], outputs=output)
model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 14)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32)        │     40,864 │ input_layer_7[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 32)        │          0 │ sequential[3][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 32)        │     41,376 │ input_layer_8[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_1 (Dot)         │ (None, 1)         │          0 │ lambda_1[0][0],   │
│                     │                   │            │ sequential_1[1][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 82,240 (321.25 KB)

 Trainable params: 82,240 (321.25 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
tf.random.set_seed(1) 
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate = 0.01) 
model.compile(optimizer = opt, loss = cost_fn)

In [35]:
tf.random.set_seed(1) 
model.fit([user_train[:, u_s:], item_train[:, i_s:]], y_train, epochs=30)

Epoch 1/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 467us/step - loss: 0.1037  
Epoch 2/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 469us/step - loss: 0.1033
Epoch 3/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 470us/step - loss: 0.1030
Epoch 4/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 469us/step - loss: 0.1031
Epoch 5/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 480us/step - loss: 0.1029
Epoch 6/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 472us/step - loss: 0.1026
Epoch 7/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 491us/step - loss: 0.1023
Epoch 8/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 462us/step - loss: 0.1023
Epoch 9/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 471us/step - loss: 0.1021
Epoch 10/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 462us/step - loss: 0.1021
Epoch 11/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 467us/step - loss: 0.1022
Epoch 12/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 490us/step - loss: 0.1020
Epoch 13/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 469us/step - loss: 0.1018
Epoch 14/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 472us/step - loss: 0.1018

In [36]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], y_test)

319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 240us/step - loss: 0.1099


0.10645482689142227

## 5 - Predictions

## 5.1 - Predictions for a new user

In [37]:
new_user_id = 5000
new_rating_ave = 0.0
new_action = 0.0
new_adventure = 5.0
new_animation = 0.0
new_childrens = 0.0
new_comedy = 0.0
new_crime = 0.0
new_documentary = 0.0
new_drama = 0.0
new_fantasy = 5.0
new_horror = 0.0
new_mystery = 0.0
new_romance = 0.0
new_scifi = 0.0
new_thriller = 0.0
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

In [38]:
def gen_user_vecs(user_vec, num_items): 
    user_vecs = np.tile(user_vec, (num_items, 1))
    return user_vecs

In [41]:
## generate and replicate the user vector to match the number movies in the dataset
user_vecs= gen_user_vecs(user_vec, len(item_vecs))

suser_vecs = scalerUser.transform(user_vecs) 
sitem_vecs = scalerItem.transform(item_vecs)

y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

# unscale y prediction
y_pu = scalerTarget.inverse_transform(y_p) 

# sort the results, highest prediction first 
sorted_index = np.argsort(-y_pu, axis = 0).reshape(-1).tolist()
sorted_ypu = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index]

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 701us/step


In [42]:
def get_user_vecs(user_id, user_train, item_vecs, user_to_genre):
    """ given a user_id, return:
        user train/predict matrix to match the size of item_vecs
        y vector with ratings for all rated movies and 0 for others of size item_vecs """

    if not user_id in user_to_genre:
        print("error: unknown user id")
        return None
    else:
        user_vec_found = False
        for i in range(len(user_train)):
            if user_train[i, 0] == user_id:
                user_vec = user_train[i]
                user_vec_found = True
                break
        if not user_vec_found:
            print("error in get_user_vecs, did not find uid in user_train")
        num_items = len(item_vecs)
        user_vecs = np.tile(user_vec, (num_items, 1))

        y = np.zeros(num_items)
        for i in range(num_items):  # walk through movies in item_vecs and get the movies, see if user has rated them
            movie_id = item_vecs[i, 0]
            if movie_id in user_to_genre[user_id]['movies']:
                rating = user_to_genre[user_id]['movies'][movie_id]
            else:
                rating = 0
            y[i] = rating
    return(user_vecs, y)

## 5.2 - Predictions for an existing user 

In [43]:
uid = 2

user_vecs, y_vecs = get_user_vecs(uid, user_train_unscaled, item_vecs, user_to_genre) 

suser_vecs = scalerUser.transform(user_vecs) 
sitem_vecs = scalerUser.transform(item_vecs) 

#make a prediction 
# make a prediction
y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

# unscale y prediction 
y_pu = scalerTarget.inverse_transform(y_p)

# sort the results, highest prediction first
sorted_index = np.argsort(-y_pu,axis=0).reshape(-1).tolist()  #negate to get largest rating first
sorted_ypu   = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index]  #using unscaled vectors for display
sorted_user  = user_vecs[sorted_index]
sorted_y     = y_vecs[sorted_index]

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 467us/step


## 5.3 - Finding Similar Items

The neural network above produces two feature vectors, a user feature vector $v_u$, and a movie feature vector, $v_m$. These are 32 entry vectors whose values are difficult to interpret. However, similar items will have similar vectors. This information can be used to make recommendations. For example, if a user has rated "Toy Story 3" highly, one could recommend similar movies by selecting movies with similar movie feature vectors.

A similarity measure is the squared distance between the two vectors $ \mathbf{v_m^{(k)}}$ and $\mathbf{v_m^{(i)}}$ :
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$

In [44]:
def sq_dist(a, b): 
    d = np.sum((a-b)**2) 

    return d

In [45]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1):0.3f}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2):0.3f}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3):0.3f}")

squared distance between a1 and b1: 0.000
squared distance between a2 and b2: 0.030
squared distance between a3 and b3: 2.000


In [48]:
from tensorflow.keras.layers import Lambda

input_item_m = tf.keras.layers.Input(shape=(num_item_features,))
vm_m = item_NN(input_item_m)

# ✅ Use a Lambda layer to wrap tf.linalg.l2_normalize
vm_m = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(vm_m)

model_m = tf.keras.Model(input_item_m, vm_m)
model_m.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 32)             │        41,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_2 (Lambda)               │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,376 (161.62 KB)

 Trainable params: 41,376 (161.62 KB)

 Non-trainable params: 0 (0.00 B)

In [49]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 802us/step
size of all predicted movie feature vectors: (847, 32)


In [50]:
count = 50  # number of movies to display
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    disp.append( [movie_dict[movie1_id]['title'], movie_dict[movie1_id]['genres'],
                  movie_dict[movie2_id]['title'], movie_dict[movie1_id]['genres']]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow")
table

movie1,genres,movie2,genres
Save the Last Dance (2001),Drama|Romance,Silent Hill (2006),Drama|Romance
"Wedding Planner, The (2001)",Comedy|Romance,Down with Love (2003),Comedy|Romance
Hannibal (2001),Horror|Thriller,X-Men Origins: Wolverine (2009),Horror|Thriller
Saving Silverman (Evil Woman) (2001),Comedy|Romance,Hostel (2005),Comedy|Romance
Down to Earth (2001),Comedy|Fantasy|Romance,"Legally Blonde 2: Red, White & Blonde (2003)",Comedy|Fantasy|Romance
"Mexican, The (2001)",Action|Comedy,Quantum of Solace (2008),Action|Comedy
15 Minutes (2001),Thriller,Lucy (2014),Thriller
Enemy at the Gates (2001),Drama,Avengers: Age of Ultron (2015),Drama
Heartbreakers (2001),Comedy|Crime|Romance,Alexander (2004),Comedy|Crime|Romance
Spy Kids (2001),Action|Adventure|Children|Comedy,Salt (2010),Action|Adventure|Children|Comedy
